# 1. Data: NDT cells and BEV images

The cells and images of every sequence are built once, on an Azure VM (`scripts/azure_build.sh`), and kept in an
Azure Blob container. This notebook prints the command for the VM and copies the result to Google Drive, where
notebooks 2 (training) and 3 (evaluation) find it:

```
KITTI zip (range requests) --scripts/azure_build.sh, Azure VM--> blob container kitti/<method>/<variant>/*.tar
    --scripts/fetch_from_azure.py--> MyDrive/lpr/<method>/<variant>/  (config.json, cells_<seq>.tar | imgs_<seq>.tar, stats_<seq>.csv)
```

A **variant** is one set of preprocessing parameters (`NDT_DEFAULTS` / `BEV_DEFAULTS` in `src/lpr/experiments.py`);
its folder name ends with the hash of the parameters, so Colab and the VM agree on it.

Optional sections: building NDT cells in Colab instead of Azure (5), the pixel check of our BEV images against the
official script (6), and the ground check of the report (7: NDT cells with and without ground removal).

Set cell 1, then **Runtime -> Run all**. No GPU is needed.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/dov85/lidar-place-recognition"
CODE_ON_DRIVE = ""                      # or the path of this code folder copied to Drive (then REPO_URL is not used)
CODE = "/content/lpr"                   # the code on this machine
DRIVE = "/content/drive/MyDrive/lpr"    # everything that is kept: variants, runs, evaluations, tables
KITTI = "/content/kitti"                # poses (and raw scans when needed)

METHOD = "ndt"          # "ndt": NDT cells (NDT-Transformer and the own network), "bev": BEV images (BEVPlace++)
VARIANT_TAG = ""        # name for a non-default variant ("" = date + hash)
VARIANT_PARAMS = dict() # only what differs from the defaults, e.g. dict(ground="none") or dict(beams=16)
SEQS = ["00", "02", "05", "06", "08"]
BEAMS = []              # also the 32 / 16-beam variants of this variant (sparsity), e.g. [32, 16]
ROT_COPIES = 0          # ndt: N > 0 also the N point-rotated copies of the training frames (runs with rot_points=N)

BUILD_HERE = False      # ndt: build missing sequences in Colab instead of on Azure (about 0.5 s per scan)
CHECK_BEV = False       # section 6: our BEV images against the official script, pixel by pixel (3 scans)
GROUND_CHECK = False    # section 7: NDT cells with and without ground removal (report, ground removal)
GROUND_SEQS = ["00", "02", "05", "06", "08"]
N_SCANS = 300           # random scans per sequence for the ground check
PREVIEWS = 6            # scans with a picture

## 2. Drive, code, poses

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
import os, sys, json, glob, subprocess, importlib
os.makedirs(DRIVE, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "no GPU"
if not os.path.exists(CODE):
    if CODE_ON_DRIVE:
        subprocess.run(["cp", "-r", CODE_ON_DRIVE, CODE], check=True)
    else:
        url = REPO_URL
        try:                            # a private repository: a token in Colab Secrets as GITHUB_TOKEN
            url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, CODE], check=True)
elif os.path.isdir(f"{CODE}/.git"):
    subprocess.run(["git", "-C", CODE, "pull", "-q"])
%cd {CODE}
!bash scripts/setup_third_party.sh
!pip install -q -r requirements.txt
sys.path.insert(0, f"{CODE}/src")
import lpr.experiments as ex, lpr.evals as ev, lpr.colab as cl
for m in (ex, ev, cl):
    importlib.reload(m)
try:                                    # container SAS URL (read + list) in Colab Secrets; never in a notebook
    SAS = userdata.get("AZURE_SAS")
except Exception:
    SAS = None
    print("no AZURE_SAS in Colab Secrets: only data that is already on Drive is used")
missing_poses = [s for s in ev.ALL_SEQS if not os.path.exists(f"{KITTI}/poses/{s}.txt")]
if missing_poses:
    !python scripts/download_kitti.py --out {KITTI} --seqs {" ".join(missing_poses)} --poses_only

## 3. Variant name and the commands for the Azure VM
Once per storage account: a private container `kitti`, and a SAS token for it with permissions read, add, create,
write, list (it works like a password: keep it in Colab Secrets as `AZURE_SAS`). Once per VM (Ubuntu, 4 cores are
enough): `git clone <this repository> && cd lidar-place-recognition`. Then run the printed commands on the VM inside
`tmux` (the build keeps running if SSH disconnects; `tail ~/work/build.log`). Raw scans already in the container are
reused, so a new variant never downloads KITTI again.

In [ ]:
VARIANT, FULL, exists = ex.resolve_variant(DRIVE, METHOD, VARIANT_PARAMS, VARIANT_TAG)
VDIR = f"{DRIVE}/{METHOD}/{VARIANT}"
print("variant:", VARIANT, "(on Drive)" if exists else "(new)")
print(json.dumps(FULL))
args = " ".join(f"--{k} {v}" for k, v in FULL.items() if v != ex.method_defaults(METHOD)[0][k])
cmd = lambda variant, a, extra="": (f'METHOD={METHOD} VARIANT={variant}' + (f' ARGS="{a}"' if a else "")
                                     + f' CELL_SEQS="{" ".join(SEQS)}"{extra} bash scripts/azure_build.sh "<SAS URL>"')
print("\nOn the Azure VM, in the repository folder (git pull first):")
print(" ", cmd(VARIANT, args))
SPARSE = {}
for b in BEAMS:
    name, p = ex.sparse_variant(METHOD, VARIANT, FULL, b)
    SPARSE[b] = name
    print(" ", cmd(name, " ".join(f"--{k} {v}" for k, v in p.items() if v != ex.method_defaults(METHOD)[0][k])))
if METHOD == "ndt" and ROT_COPIES:
    print(" ", f'METHOD=ndt VARIANT={VARIANT}' + (f' ARGS="{args}"' if args else "")
          + f' CELL_SEQS="" ROT_COPIES={ROT_COPIES} ROT_SEQS=00 bash scripts/azure_build.sh "<SAS URL>"')
if GROUND_CHECK:
    print(" ", f'SEQS="{" ".join(GROUND_SEQS)}" N_SCANS={N_SCANS} bash scripts/azure_ground_check.sh "<SAS URL>"')

## 4. Blob container -> Drive
Copies what is built on Azure and not on Drive yet (and checks that the parameters built on Azure are the ones of
the variant). Sequences that are not built yet are listed; build them on the VM (section 3) and run this again.

In [ ]:
if SAS:
    missing = [s for s in SEQS if not os.path.exists(f"{VDIR}/{ex.archive_name(METHOD, s)}")]
    if missing:
        cl.run(["scripts/fetch_from_azure.py", "--sas_url", SAS, "--method", METHOD, "--variant", VARIANT,
                "--seqs", *missing, "--drive", DRIVE])
    if BEAMS:
        cl.sparse_to_machine(DRIVE, METHOD, VARIANT, BEAMS, SEQS, SAS)
    if METHOD == "ndt" and ROT_COPIES and not os.path.exists(f"{VDIR}/{ex.rot_archive_name('00', ROT_COPIES)}"):
        cl.run(["scripts/fetch_from_azure.py", "--sas_url", SAS, "--method", "ndt", "--variant", VARIANT,
                "--seqs", "00", "--rot_copies", ROT_COPIES, "--drive", DRIVE])
print("on Drive:", [s for s in SEQS if os.path.exists(f"{VDIR}/{ex.archive_name(METHOD, s)}")])

## 5. Optional (`BUILD_HERE`, ndt): build missing sequences in Colab
Raw scans from the blob container (`raw/kitti_seq<seq>_raw.tar`) if there, otherwise downloaded from KITTI (range
requests into the zip, only the needed sequence). Then crop, ground removal and G on every scan
(`scripts/extract_ndt.py`, with the coverage checks of `src/lpr/ndt_check.py`), and the tar on Drive.

In [ ]:
def raw_scans(seq):
    """Raw scans of one sequence in KITTI: blob container if there, otherwise KITTI itself."""
    vel = f"{KITTI}/sequences/{seq}/velodyne"
    if os.path.isdir(vel) and os.listdir(vel):
        return
    tmp = f"/content/raw_{seq}.tar"
    if SAS:
        base, token = SAS.split("?", 1)
        !curl -sf -o {tmp} "{base.rstrip('/')}/raw/kitti_seq{seq}_raw.tar?{token}" && tar -xf {tmp} -C {KITTI} && rm -f {tmp}
    if not (os.path.isdir(vel) and os.listdir(vel)):
        !python scripts/download_kitti.py --out {KITTI} --seqs {seq} --threads 48

if not (BUILD_HERE and METHOD == "ndt"):
    print("skipped (BUILD_HERE = False or METHOD != 'ndt')")
else:
    ndt_args = " ".join(f"--{k} {v}" for k, v in FULL.items())
    local = f"/content/ndt/{VARIANT}"
    os.makedirs(VDIR, exist_ok=True); os.makedirs(local, exist_ok=True)
    if os.path.exists(f"{VDIR}/config.json"):        # existing variant, new sequence: keep its history
        !cp {VDIR}/config.json {local}/config.json
    for s in SEQS:
        if os.path.exists(f"{VDIR}/cells_{s}.tar"):
            continue
        raw_scans(s)
        !python scripts/extract_ndt.py --kitti_root {KITTI} --seq {s} --out_dir {local} --variant {VARIANT} {ndt_args}
        !tar -cf {VDIR}/cells_{s}.tar -C {local} {s} {s}_stats.csv config.json
        !cp {local}/{s}_stats.csv {VDIR}/stats_{s}.csv
        if os.path.isdir(f"{local}/{s}_previews"):
            !mkdir -p {VDIR}/previews_{s} && cp {local}/{s}_previews/*.png {VDIR}/previews_{s}/
        !cp {local}/config.json {VDIR}/config.json

## 6. Optional (`CHECK_BEV`): our BEV images = the official script's
`src/lpr/bev.py` against `third_party/BEVPlace2/datasets/gen_bev_images.py` on 3 scans of sequence 00.
Expected last line: `IDENTICAL`.

In [ ]:
if CHECK_BEV:
    !pip install -q open3d
    !python scripts/download_kitti.py --out /content/kitti_check --seqs 00 --first 3
    scans = " ".join(f"/content/kitti_check/sequences/00/velodyne/{i:06d}.bin" for i in range(3))
    !python tests/test_bev_matches_official.py --scans {scans}
else:
    print("skipped (CHECK_BEV = False)")

## 7. Optional (`GROUND_CHECK`): NDT cells with and without ground removal
For `N_SCANS` random scans per sequence, the cells are built twice from the same cropped scan
(`scripts/ground_coverage.py`): with Patchwork++ ground removal (the default) and with the ground kept. Per scan:
% of the points within 0.8 m of a cell mean (all / non-ground / ground), % of the cells on the ground, voxel size
of G. Sequences already checked on the VM (`scripts/azure_ground_check.sh`) are copied from the blob container,
the others run here. Results: `MyDrive/lpr/ground_check/<tag>/`.

In [ ]:
import shutil, tarfile, requests
import pandas as pd
from IPython.display import Image, display

def blob_get(path, dest):
    base, token = SAS.split("?", 1)
    with requests.get(f"{base.rstrip('/')}/{path}?{token}", stream=True, timeout=60) as r:
        if r.status_code != 200:
            return False
        with open(dest, "wb") as f:
            shutil.copyfileobj(r.raw, f, length=16 << 20)
    return True

if not GROUND_CHECK:
    print("skipped (GROUND_CHECK = False)")
else:
    GP = {k: v for k, v in VARIANT_PARAMS.items() if k != "ground"} if METHOD == "ndt" else {}
    TAG = ex.ground_check_tag(GP, N_SCANS)           # the folder name of scripts/azure_ground_check.sh
    OUT = f"{DRIVE}/ground_check/{TAG}"
    os.makedirs(OUT, exist_ok=True)
    done = lambda s: os.path.exists(f"{OUT}/ground_coverage_{s}_summary.csv")
    for s in GROUND_SEQS:                             # results of the VM
        if done(s) or not SAS or not blob_get(f"ground_check/{TAG}/ground_coverage_{s}_summary.csv", f"/content/gc_{s}.csv"):
            continue
        for f in [f"ground_coverage_{s}.csv", f"ground_coverage_{s}_params.json"]:
            blob_get(f"ground_check/{TAG}/{f}", f"{OUT}/{f}")
        if blob_get(f"ground_check/{TAG}/ground_previews_{s}.tar", f"/content/gp_{s}.tar"):
            with tarfile.open(f"/content/gp_{s}.tar") as t:
                t.extractall(OUT)
        shutil.copy(f"/content/gc_{s}.csv", f"{OUT}/ground_coverage_{s}_summary.csv")   # last: marks it done
    for s in GROUND_SEQS:                             # the rest here
        if not done(s):
            raw_scans(s)
            !python scripts/ground_coverage.py --kitti_root {KITTI} --seq {s} --n_scans {N_SCANS} --previews {PREVIEWS} --out_dir {OUT} --params '{json.dumps(GP)}'
    S = pd.concat([pd.read_csv(f"{OUT}/ground_coverage_{s}_summary.csv", dtype={"seq": str}) for s in GROUND_SEQS if done(s)])
    S.to_csv(f"{OUT}/summary_all.csv", index=False)
    KEY = ["ground_share", "cov_all", "cov_nonground", "cov_ground", "cells_ground", "voxel", "g_seconds"]
    display(S.pivot_table(index="seq", columns="mode", values=KEY)[KEY].round(2))
    for png in sorted(glob.glob(f"{OUT}/ground_previews_*/*.png"))[:PREVIEWS]:
        display(Image(png, width=1400))